# Notebook 02: Training con MLflow

Entrena modelos de clasificación de especialidades médicas
con tracking completo usando MLflow.

In [2]:
import sys
import os

# Obtén la ruta absoluta del directorio raíz del proyecto (tu_proyecto/)
ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Añade la carpeta 'src' al sys.path
sys.path.append(os.path.join(ruta_raiz, "src"))

# Ahora importa config
import config


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# MLflow
import mlflow
import mlflow.pytorch

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


In [5]:
# Datasets y métricas
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch

print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"Número de GPUs detectadas: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Nombre de la GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

CUDA disponible: True
Número de GPUs detectadas: 1
Nombre de la GPU: NVIDIA GeForce GTX 1650
Usando dispositivo: cuda


## Configuración de MLflow Tracking


In [6]:
# Establecer URI de tracking (local)
mlflow.set_tracking_uri("file:../mlruns")

# Crear/usar experimento
experiment_name = "medical-nlp-classification"
mlflow.set_experiment(experiment_name)

# Obtener información del experimento
experiment = mlflow.get_experiment_by_name(experiment_name)
print(f"✅ MLflow configurado")
print(f"📊 Experimento: {experiment_name}")
print(f"📁 Tracking URI: {mlflow.get_tracking_uri()}")
print(f"🆔 Experiment ID: {experiment.experiment_id}")
print(f"\n💡 Para ver la UI de MLflow, ejecuta en terminal:")
print(f"   cd .. && mlflow ui")
print(f"   Luego abre: http://127.0.0.1:5000")

✅ MLflow configurado
📊 Experimento: medical-nlp-classification
📁 Tracking URI: file:../mlruns
🆔 Experiment ID: 780220363043288146

💡 Para ver la UI de MLflow, ejecuta en terminal:
   cd .. && mlflow ui
   Luego abre: http://127.0.0.1:5000


## Cargar datos procesados

In [20]:
train_path = Path("../data/processed/train.csv")
val_path = Path("../data/processed/val.csv")
test_path = Path("../data/processed/test.csv")

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"✅ Datos cargados")
print(f"📊 Train: {len(train_df)} muestras")
print(f"📊 Test: {len(test_df)} muestras")
print(f"📊 Clases: {train_df['specialty_clean'].nunique()}")

print(f"\n📈 Distribución de clases en Train:")
print(train_df['specialty_clean'].value_counts().head(10))

✅ Datos cargados
📊 Train: 9324 muestras
📊 Test: 2073 muestras
📊 Clases: 20

📈 Distribución de clases en Train:
specialty_clean
 Surgery                       2106
 Consult - History and Phy.    1126
Others                          952
 Orthopedic                     781
 Cardiovascular / Pulmonary     669
 General Medicine               484
 Neurology                      440
 Radiology                      358
 Gastroenterology               344
 Obstetrics / Gynecology        291
Name: count, dtype: int64


## Preparar encoding de labels


In [8]:
label2id = {label: idx for idx, label in enumerate(train_df['specialty_clean'].unique())}
id2label = {idx: label for label, idx in label2id.items()}

print(f"\n✅ Encoding de labels preparado")
print(f"📊 Número de clases: {len(label2id)}")
print(f"🔤 Ejemplo label2id: {dict(list(label2id.items() )[:10])}")
print(f"🔤 Ejemplo id2label: {dict(list(id2label.items())[-10:])}")


✅ Encoding de labels preparado
📊 Número de clases: 20
🔤 Ejemplo label2id: {'Others': 0, ' Radiology': 1, ' Surgery': 2, ' Urology': 3, ' Cardiovascular / Pulmonary': 4, ' Pediatrics - Neonatal': 5, ' SOAP / Chart / Progress Notes': 6, ' Consult - History and Phy.': 7, ' Neurosurgery': 8, ' General Medicine': 9}
🔤 Ejemplo id2label: {10: ' Neurology', 11: ' Gastroenterology', 12: ' Obstetrics / Gynecology', 13: ' Orthopedic', 14: ' ENT - Otolaryngology', 15: ' Emergency Room Reports', 16: ' Nephrology', 17: ' Ophthalmology', 18: ' Discharge Summary', 19: ' Hematology - Oncology'}


In [21]:
train_df['label'] = train_df['specialty_clean'].map(label2id)
val_df['label'] = train_df['specialty_clean'].map(label2id)
test_df['label'] = test_df['specialty_clean'].map(label2id)

print(f"\n✅ Columna 'label' añadida a los datasets")
print(f"📊 Ejemplo de train_df:")
print(train_df.sample(5))


✅ Columna 'label' añadida a los datasets
📊 Ejemplo de train_df:
                                          transcription  \
7684  S:,  XYZ is in today not feeling well for the ...   
620   ed out in the disk space.  At this time, the d...   
987   ngeur was used to take out any osteophytes and...   
7719  tibialis anterior, and extensor hallucis longu...   
7896  e secured with 16-mm titanium screws after exc...   

                     specialty_clean  original_id  chunk_id  total_chunks  \
7684   SOAP / Chart / Progress Notes         1377         0             1   
620                          Surgery          670         1             2   
987                          Surgery          253         1             4   
7719                      Orthopedic         2180         1             2   
7896                    Neurosurgery         2745         1             2   

      label  
7684      6  
620       2  
987       2  
7719     13  
7896      8  


## Cargamos tokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f"\n✅ Tokenizer '{config.MODEL_NAME}' cargado")
print(f"✓ Vocab size: {tokenizer.vocab_size}")


✅ Tokenizer 'distilbert-base-uncased' cargado
✓ Vocab size: 30522


In [11]:
sample_text = train_df.iloc[0]['transcription']
max_length = tokenizer.model_max_length
tokens = tokenizer(
    sample_text,
    max_length=max_length,
    truncation=True,
    padding='max_length',
    return_tensors='pt'
)

print(f"\n✓ Test de tokenización exitoso")
print(f"  - Input IDs shape: {tokens['input_ids'].shape}")
print(f"  - Attention mask shape: {tokens['attention_mask'].shape}")


✓ Test de tokenización exitoso
  - Input IDs shape: torch.Size([1, 512])
  - Attention mask shape: torch.Size([1, 512])


## Preparación datos para entrenamiento

In [22]:
X_train = train_df['transcription'].tolist()
y_train = train_df['label'].tolist()
X_val   = val_df['transcription'].tolist()
y_val   = val_df['label'].tolist()
X_test  = test_df['transcription'].tolist()
y_test  = test_df['label'].tolist()

print(f"\n✅ Datos preparados para entrenamiento")
print(f"📊 Train samples: {len(X_train)}")
print(f"📊 Val samples: {len(X_val)}")
print(f"📊 Test samples: {len(X_test)}")
print(f"sample X_train: {X_train[1:5][:10]}...")
print(f"sample y_train: {y_train[10:30]}")


✅ Datos preparados para entrenamiento
📊 Train samples: 9324
📊 Val samples: 1037
📊 Test samples: 2073
sample X_train: ['nt physical therapy.  She returned to the Sinai ER on 08/2009/2009 due to reported left arm pain, numbness, and weakness, which lasted 10 to 15 minutes and she reported that it felt "just like the stroke."  Brain CT on 08/2009/2009 was read as showing "mild chronic microvascular ischemic change of deep white matter," but no acute or significant interval change compared to her previous scan.  Neurological examination with Dr. Y was within normal limits, but she was admitted for a more extensive workup.  Due to left arm pain an ultrasound was completed on her left upper extremity, but it did not show deep vein thrombosis.,Followup CT on 08/10/2009 showed no significant interval change.  MRI could not be completed due to the patient\'s weight.  She was discharged on 08/11/2009 in stable condition after it was determined that this event was not neurological in origin; how

In [23]:
# ---------- 1) Crear Dataset de HuggingFace ----------
train_data = Dataset.from_dict({"text": X_train, "label": y_train})
val_data   = Dataset.from_dict({"text": X_val, "label": y_val})
test_data  = Dataset.from_dict({"text": X_test, "label": y_test})

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )

train_tokenized = train_data.map(tokenize_function, batched=True)
val_tokenized   = val_data.map(tokenize_function, batched=True)
test_tokenized  = test_data.map(tokenize_function, batched=True)

print(f"\n✅ Datasets tokenizados")
print(f"📊 Train tokenized sample:")
print(train_tokenized[0])

Map:   0%|          | 0/9324 [00:00<?, ? examples/s]

Map:   0%|          | 0/1037 [00:00<?, ? examples/s]

Map:   0%|          | 0/2073 [00:00<?, ? examples/s]


✅ Datasets tokenizados
📊 Train tokenized sample:
{'text': "DESCRIPTION OF RECORD:  ,This tracing was obtained utilizing 27 paste-on gold-plated surface disc electrodes placed according to the International 10-20 system.  Electrode impedances were measured and reported at less than 5 kilo-ohms each.,FINDINGS: , In general, the background rhythms are bilaterally symmetrical.  During the resting awake state they are composed of moderate amounts of low amplitude fast activity intermixed with moderate amounts of well-modulated 9-10 Hz alpha activity best seen posteriorly.  The alpha activity attenuates with eye opening.,During some portions of the tracing the patient enters a drowsy state in which the background rhythms are composed predominantly of moderate amounts of low amplitude fast activity intermixed with moderate amounts of low to medium amplitude polymorphic theta activity.,There is no evidence of focal slowing or paroxysmal activity.,IMPRESSION: , Normal awake and drowsy (stage I

In [24]:
# ---------- 3) Data collator y formato tensor ----------
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenización lista. Ejemplo:")
print(train_tokenized[0])

Tokenización lista. Ejemplo:
{'label': tensor(0), 'input_ids': tensor([  101,  6412,  1997,  2501,  1024,  1010,  2023, 16907,  2001,  4663,
        16911,  2676, 19351,  1011,  2006,  2751,  1011,  5127,  2094,  3302,
         5860, 28688,  2015,  2872,  2429,  2000,  1996,  2248,  2184,  1011,
         2322,  2291,  1012, 28688, 17727, 29605,  2015,  2020,  7594,  1998,
         2988,  2012,  2625,  2084,  1019, 11382,  4135,  1011,  2821,  5244,
         2169,  1012,  1010,  9556,  1024,  1010,  1999,  2236,  1010,  1996,
         4281, 17900,  2024, 17758,  2135, 23476,  1012,  2076,  1996,  8345,
         8300,  2110,  2027,  2024,  3605,  1997,  8777,  8310,  1997,  2659,
        22261,  3435,  4023,  6970,  4328, 19068,  2007,  8777,  8310,  1997,
         2092,  1011, 16913,  8898,  1023,  1011,  2184, 22100,  6541,  4023,
         2190,  2464, 15219,  2135,  1012,  1996,  6541,  4023,  2012,  6528,
        20598,  2015,  2007,  3239,  3098,  1012,  1010,  2076,  2070,  8810,
 

In [25]:
import os

# Crear carpeta donde guardar los datasets tokenizados
os.makedirs("../data/tokenized", exist_ok=True)

# Guardar en formato HuggingFace (.arrow)
train_tokenized.save_to_disk("../data/tokenized/train")
val_tokenized.save_to_disk("../data/tokenized/val")
test_tokenized.save_to_disk("../data/tokenized/test")

print("✅ Datasets tokenizados guardados en 'data/tokenized/'")

# Verificación opcional: cargar uno y ver que todo esté bien
from datasets import load_from_disk
check = load_from_disk("../data/tokenized/train")
print(f"Verificación OK: {len(check)} muestras en train")


Saving the dataset (0/1 shards):   0%|          | 0/9324 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1037 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2073 [00:00<?, ? examples/s]

✅ Datasets tokenizados guardados en 'data/tokenized/'
Verificación OK: 9324 muestras en train
